# National Open Economy - Prospective

## 0. Initialize context

In [ ]:
%run ../0-context/context.py

In [ ]:
dir_input = "../../data/1-EEIO_foundations/2-national_open_economy/1-notebook_input/"
dir_output = "../../data/1-EEIO_foundations/2-national_open_economy/2-notebook_output/"

### 0.1. Load accounts

In [ ]:
from matmat.core.accounts import builder as accounts_builder
path_to_accounts = os.path.join(dir_input, "accounts")
accounts_director = accounts_builder.get_director(reset=True)
accounts = accounts_director.make_from_path(path=path_to_accounts)
accounts.calculate()

## 1. Shock on the investment matrix (Y_k)

### 1.1. Initialize system shock consistent with accounts system

In [ ]:
from matmat.core.shocks.system import builder as system_shock_builder
system_shock_director = system_shock_builder.get_director(reset=True)
system_shock_director.set_regions(accounts.system.regions)
system_shock_director.set_sectors(accounts.system.sectors)
system_shock_director.set_final_demand_categories(accounts.system.final_demand_categories)

In [ ]:
# Initialize a system shock for investment matrix variation
system_shock = system_shock_director.make_shock_y_k()

In [ ]:
system_shock.id

In [ ]:
# Update identity fields
system_shock.id.base_year = accounts.system.id.base_year
system_shock.id.proj_year = 2050
system_shock.id.scenario_name="formation"

**Exercise:** Display investment shock matrix

In [ ]:
# Display dY_k
system_shock.dY_k.df

### 1.2. Define the relative variations

**Exercise:** Set a 20% decrease to investment matrix to the sector 'Construction' for category 'Services'

In [ ]:
system_shock.dY_k.set_values(0.0)
system_shock.dY_k.set_value_at(
    value=-0.2,
    rows_filters={"sector": ["Construction"]},
    columns_filters={"category": ["Services"]}, 
)
system_shock.dY_k.df.xs(axis=1, level="category", key="Services", drop_level=False)

### 1.3. Shock the system

**Exercise:** Apply the shock to the system

In [ ]:
# Duplicate references
accounts_calibrated = accounts.copy()
accounts_projected = accounts.copy()
# Apply the system shock to the accounts system
# help: use the method reset_for_shock() on the accounts, shock() on the system
accounts_projected.reset_for_shock()
accounts_projected.system.shock(system_shock)

**Exercise:** Display the changes on investment data Y_k

In [ ]:
# Display Y_k from calibrated system
accounts.system.Y_k.df.xs(axis=0, level="sector", key="Construction").xs(axis=1, level="category", key="Services", drop_level=False)

In [ ]:
# Display Y_k from projected system
accounts_projected.system.Y_k.df.xs(axis=0, level="sector", key="Construction").xs(axis=1, level="category", key="Services", drop_level=False)

### 1.3. Recalculate accounts

We now want to propagate this modification to the whole accounts, by recalculating:
- the system
- the extensions

**Exercise:** Calculate the accounts

In [ ]:
# Calculate the projected accounts
accounts_projected.calculate()

**Exercise:** Show the modification of dom_ghg_combustion d_cba in projected accounts

In [ ]:
accounts_calibrated.dom_ghg_combustion.d_cba.df

In [ ]:
# Display dom_ghg_combustion d_cba from calibrated accounts
accounts_calibrated.dom_ghg_combustion.d_cba.df.xs(axis=0, level="sector", key="Fossil fuels").xs(axis=1, level="category", key="Construction", drop_level=False)

In [ ]:
# Display dom_ghg_combustion d_cba from projected accounts
accounts_projected.dom_ghg_combustion.d_cba.df.xs(axis=0, level="sector", key="Fossil fuels").xs(axis=1, level="category", key="Construction")

### 1.4. Save accounts and shock

In [ ]:
# Use the method save_data() to save the accounts to local directory
accounts_projected.save_to_path(os.path.join(dir_output, "accounts", "accounts_shocked_by_dY_k"))

In [ ]:
# Use the method save_data() to save the accounts to local directory
system_shock.save_to_path(os.path.join(dir_output, "shocks", "shock_Y_k", "system"))

## 2. Shock on the capital coefficients matrix (K)

### 1.1. Initialize system shock consistent with accounts system

In [ ]:
from matmat.core.shocks.system import builder as system_shock_builder
system_shock_director = system_shock_builder.get_director(reset=True)
system_shock_director.set_regions(accounts.system.regions)
system_shock_director.set_sectors(accounts.system.sectors)
system_shock_director.set_final_demand_categories(accounts.system.final_demand_categories)

In [ ]:
# Initialize a system shock for K variation
system_shock = system_shock_director.make_shock_k()

In [ ]:
system_shock.id

In [ ]:
# Update identity fields
system_shock.id.base_year = accounts.system.id.base_year
system_shock.id.proj_year = 2050
system_shock.id.scenario_name="formation"

**Exercise:** Display capital coefficients shock matrices

In [ ]:
# Display dK
system_shock.dK.df

### 1.2. Define the relative variations

**Exercise:** Set a 20% decrease to capital coefficients matrix to the sector 'Construction' for category 'Services'

In [ ]:
system_shock.dK.set_values(0.0)
system_shock.dK.set_value_at(
    value=-0.2,
    rows_filters={"sector": ["Construction"]},
    columns_filters={"category": ["Services"]}, 
)
system_shock.dK.df

### 1.3. Shock the system

**Exercise:** Switch the system calcul strategy to 'endo_invest_matrix'

In [ ]:
accounts_calibrated = accounts.copy()
accounts_calibrated.system.calcul = "endo_invest_matrix"

**Exercise:** Apply the shock to the system

In [ ]:
# Duplicate references
accounts_projected = accounts_calibrated.copy()
# Apply the system shock to the accounts system
# help: use the method reset_for_shock() on the accounts, shock() on the system
accounts_projected.reset_for_shock()
accounts_projected.system.shock(system_shock)

**Exercise:** Display the changes on capital coefficients data K

In [ ]:
# Display K from calibrated system
accounts_calibrated.system.K.df.xs(axis=0, level="sector", key="Construction").xs(axis=1, level="category", key="Services", drop_level=False)

In [ ]:
# Display K from calibrated system
accounts_projected.system.K.df.xs(axis=0, level="sector", key="Construction").xs(axis=1, level="category", key="Services", drop_level=False)

### 1.3. Recalculate accounts

We now want to propagate this modification to the whole accounts, by recalculating:
- the system
- the extensions

**Exercise:** Calculate the accounts

In [ ]:
# Calculate the projected accounts
accounts_projected.calculate()

**Exercise:** Show the modification of dom_ghg_combustion d_cba in projected accounts

In [ ]:
# Display dom_ghg_combustion d_cba from calibrated accounts
accounts_calibrated.dom_ghg_combustion.d_cba.df.xs(axis=0, level="sector", key="Fossil fuels", drop_level=False).xs(level="category", key="Construction", axis=1, drop_level=False)

In [ ]:
# Display dom_ghg_combustion d_cba from projected accounts
accounts_projected.dom_ghg_combustion.d_cba.df.xs(axis=0, level="sector", key="Fossil fuels", drop_level=False).xs(level="category", key="Construction", axis=1, drop_level=False)

### 1.4. Save accounts and shock

In [ ]:
# Use the method save_data() to save the accounts to local directory
accounts_projected.save_to_path(os.path.join(dir_output, "accounts", "accounts_shocked_by_dK"))

In [ ]:
# Use the method save_data() to save the accounts to local directory
system_shock.save_to_path(os.path.join(dir_output, "shocks", "shock_dK", "system"))

## 3. Shock on M_RoW

### 3.1. Initialize extension shock consistent with extension imp_ghg_emissions

In [ ]:
# Initialize an extension shock for intensities variation
from matmat.core.shocks.extension import builder as extension_shock_builder
extension_shock_director = extension_shock_builder.get_director(reset=True)
extension_shock_director.set_regions(accounts.imp_ghg_emissions.regions)
extension_shock_director.set_sectors(accounts.imp_ghg_emissions.sectors)
extension_shock_director.set_final_demand_categories(accounts.imp_ghg_emissions.final_demand_categories)
extension_shock_director.set_extension_categories(accounts.imp_ghg_emissions.extension_categories)

imp_ghg_emissions_shock = extension_shock_director.make_shock_m_row(name="imp_ghg_emissions")

In [ ]:
# Update identity fields
imp_ghg_emissions_shock.id.base_year = accounts.imp_ghg_emissions.id.base_year
imp_ghg_emissions_shock.id.proj_year = 2050
imp_ghg_emissions_shock.id.scenario_name="formation"

**Exercise:** Display dM_RoW shock matrices

In [ ]:
# Display dM_RoW
imp_ghg_emissions_shock.dM_RoW.df

### 3.2. Define the relative variations

**Exercise:** Define the variations on M_RoW
- Set a 50% decrease for category 'Industry'

In [ ]:
imp_ghg_emissions_shock.dM_RoW.set_values(0.0)
imp_ghg_emissions_shock.dM_RoW.set_value_at(
    value=-0.5,
    rows_filters=None,
    columns_filters={"category": ["Industry"]},
)
imp_ghg_emissions_shock.dM_RoW.df

### 3.3. Shock the extension imp_ghg_emissions

**Exercise:** Apply the shock to the extension

In [ ]:
# Duplicate references
accounts_calibrated = accounts.copy()
accounts_projected = accounts.copy()
# Apply the extension shock to the imp_ghg_emissions
# help: use the method reset_for_shock() on the accounts, shock() on the extension
accounts_projected.reset_for_shock()
accounts_projected.imp_ghg_emissions.shock(imp_ghg_emissions_shock)

**Exercise:** Display the changes on intensities M_RoW

In [ ]:
# Display M_RoW from calibrated imp_ghg_emissions
accounts_calibrated.imp_ghg_emissions.M_RoW.df.xs(axis=1, level="category", key="Industry")

In [ ]:
# Display M_RoW from projected imp_ghg_emissions
accounts_projected.imp_ghg_emissions.M_RoW.df.xs(axis=1, level="category", key="Industry")

### 3.3. Recalculate imp_ghg_emissions

**Exercise:** Calculate the projected imp_ghg_emissions

In [ ]:
# Calculate the projected imp_ghg_emissions
accounts_projected.calculate()

**Exercise:** Show the modification of imp_ghg_emissions d_cba in projected accounts

In [ ]:
# Display imp_ghg_emissions d_cba from calibrated accounts
accounts_calibrated.imp_ghg_emissions.d_cba.df.xs(axis=1, level="category", key="Industry", drop_level=False).xs(axis=0, level="gas", key="CO2", drop_level=False)

In [ ]:
# Display imp_ghg_emissions d_cba from projected accounts
accounts_projected.imp_ghg_emissions.d_cba.df.xs(axis=1, level="category", key="Industry", drop_level=False).xs(axis=0, level="gas", key="CO2", drop_level=False)

### 3.4. Save imp_ghg_emissions shock

In [ ]:
# Use the method save_data() to save the shock to local directory
imp_ghg_emissions_shock.save_to_path(os.path.join(dir_output, "shocks", "shock_M_RoW", "extensions"))

## 4. BONUS: Shock on S_Z

**Exercise:** Apply a shock on S_Z to the extension dom_ghg_combustion
- Initialize the proper extension shock
- Valorize it (-50% to the sector "Energy supply" in the category "Industry")
- Apply the shock to the extension
- Recalculate the extension
- Display the results in the d_cba

#### Initialize the shock

In [ ]:
# Initialize an extension shock for S_Z variation
from matmat.core.shocks.extension import builder as extension_shock_builder
extension_shock_director.set_extension_categories(accounts.dom_ghg_combustion.extension_categories)

dom_ghg_combustion_shock = extension_shock_director.make_shock_s_z(name="dom_ghg_combustion")

In [ ]:
# Update identity fields
dom_ghg_combustion_shock.id.base_year = accounts.dom_ghg_combustion.id.base_year
dom_ghg_combustion_shock.id.proj_year = 2050
dom_ghg_combustion_shock.id.scenario_name="formation"

**Exercise:**
- Valorize dS_Z (-50% to the sector "Energy supply" (rows) in the category "Industry" (columns))
- Apply the shock to the extension dom_ghg_combustion_shock
- Recalculate the extension dom_ghg_combustion_shock
- Display the results in the d_cba

In [ ]:
# Your move ;)

### 4.1. Solution

**Exercise:** Define the variations on dS_Z
- Set a 50% decrease for sector "Energy supply" and category 'Industry'

In [ ]:
dom_ghg_combustion_shock.dS_Z.set_values(0.0)
dom_ghg_combustion_shock.dS_Z.set_value_at(
    value=-0.5,
    rows_filters={"sector": ["Energy supply"]},
    columns_filters={"category": ["Industry"]},
)
dom_ghg_combustion_shock.dS_Z.df

#### Shock the extension dom_ghg_combustion

**Exercise:** Apply the shock to the extension

In [ ]:
# Duplicate references
accounts_calibrated = accounts.copy()
accounts_projected = accounts.copy()
# Apply the extension shock to the dom_ghg_combustion
# help: use the method reset_for_shock() and shock() on the extension
accounts_projected.dom_ghg_combustion.reset_for_shock()
accounts_projected.dom_ghg_combustion.shock(dom_ghg_combustion_shock)

**Exercise:** Display the changes on intensities S_Z

In [ ]:
# Display S_Z from calibrated dom_ghg_combustion
accounts_calibrated.dom_ghg_combustion.S_Z.df.xs(axis=0, level="sector", key="Energy supply").xs(axis=1, level="category", key="Industry")

In [ ]:
# Display S_Z from projected dom_ghg_combustion
accounts_projected.dom_ghg_combustion.S_Z.df.xs(axis=0, level="sector", key="Energy supply").xs(axis=1, level="category", key="Industry")

#### Recalculate dom_ghg_combustion

**Exercise:** Calculate the projected dom_ghg_combustion

In [ ]:
# Calculate the projected dom_ghg_combustion
accounts_projected.dom_ghg_combustion.calculate(accounts_projected.system)

**Exercise:** Show the modification of dom_ghg_combustion d_cba in projected accounts

In [ ]:
# Display dom_ghg_combustion d_cba from calibrated accounts
accounts_calibrated.dom_ghg_combustion.d_cba.df.xs(
    axis=0, level="sector", key="Energy supply", drop_level=False
).xs(axis=1, level="category", key="Industry", drop_level=False)

In [ ]:
# Display dom_ghg_combustion d_cba from calibrated accounts
accounts_projected.dom_ghg_combustion.d_cba.df.xs(
    axis=0, level="sector", key="Energy supply", drop_level=False
).xs(axis=1, level="category", key="Industry", drop_level=False)

#### Save dom_ghg_combustion shock

In [ ]:
# Use the method save_data() to save the shock to local directory
dom_ghg_combustion_shock.save_to_path(os.path.join(dir_output, "shocks", "shock_dS_Z", "extensions"))